# Quickstart 2 — design matrices and linear solves

The idea behind the whole open approach: **at a fixed orbit shape (P, e, τ)
the model is linear in its amplitudes**. The RV curve is linear in
(γ, K cos ω, K sin ω); the along-scan model is linear in the four Thiele–Innes
amplitudes plus the five single-star parameters. So a fit is a small
generalised-least-squares solve per shape, and only the shape needs searching.

This notebook shows the design matrices, the solves, and the ranking score they
return — including the one place where that score must not be used.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from orblet.constants import DAYS_PER_KEPLER_YEAR
from orblet.simulate.bundles import load_simulated_inputs
from orblet import prepare_rv_for_orbit, resolve_epochs_mjd

bundle = load_simulated_inputs(seed=1)
truth = bundle.truth
astro = bundle.astro_data
prepared = prepare_rv_for_orbit(bundle.rv_data, time_scale="gaia_obmt")

EPOCH_REF = float(truth.t_ref_mjd)
P_YR = truth.P_days / DAYS_PER_KEPLER_YEAR
TAU = ((truth.tp_mjd - EPOCH_REF) / truth.P_days) % 1.0

t_rv, rv, rv_err = prepared["epochs_mjd"], prepared["rv"], prepared["rv_err"]
t_mjd = resolve_epochs_mjd(astro)
psi = np.asarray(astro["scan_angle"], dtype=float)
pf = np.asarray(astro["parallax_factor_al"], dtype=float)
d_obs = np.asarray(astro["centroid_pos"], dtype=float)
sigma = np.asarray(astro["centroid_pos_err"], dtype=float)

## RV: three columns, one solve

`rv_design_matrix` builds the columns [1, cos ν + e, −sin ν] at a shape;
`linear_solve_rv` returns the amplitudes β = (γ, K cos ω, K sin ω), their
covariance, and a **ranking score** `logL_marginal` (the amplitudes integrated
out under a flat prior). `recover_K` and `recover_omega` undo the split.

In [ ]:
from orblet import rv_design_matrix, linear_solve_rv, recover_K, recover_omega

X = rv_design_matrix(t_rv, period_yr=P_YR, ecc=truth.e, tau=TAU, epoch_ref_mjd=EPOCH_REF)
sol = linear_solve_rv(rv, rv_err, X)
print("design matrix shape:", X.shape)
print(f"gamma = {sol.beta[0]:.2f} km/s, K = {recover_K(sol.beta):.2f} km/s (truth {truth.K1_kms:.2f}), "
      f"omega = {np.degrees(recover_omega(sol.beta)):.1f} deg (truth {np.degrees(truth.omega_rad):.1f})")
print(f"chi2/dof = {sol.chi2:.1f}/{sol.dof}; logL_marginal = {sol.logL_marginal:.1f}")

# The ranking score as a function of the period, everything else at the truth:
# this is the RV periodogram of the open approach (comb-like near the truth).
periods = np.linspace(0.8 * truth.P_days, 1.2 * truth.P_days, 400)
score = [linear_solve_rv(rv, rv_err, rv_design_matrix(t_rv, period_yr=P / DAYS_PER_KEPLER_YEAR,
         ecc=truth.e, tau=TAU, epoch_ref_mjd=EPOCH_REF)).logL_marginal for P in periods]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(periods, score); ax.axvline(truth.P_days, color="k", lw=0.8)
ax.set_xlabel("period (days)"); ax.set_ylabel("logL_marginal")
plt.show()

In [ ]:
from orblet import best_linear_params_rv, semi_amplitude_kms

# The same solve packaged with covariance-correct amplitude draws (period in DAYS here).
blp = best_linear_params_rv(truth.P_days, truth.e, TAU, epochs_mjd=t_rv, rv=rv, rv_err=rv_err,
                            epoch_ref_mjd=EPOCH_REF, draw=500)
K_draws = np.array([recover_K(b) for b in blp.draws])
print(f"K = {np.median(K_draws):.2f} +/- {np.std(K_draws):.2f} km/s from {blp.draws.shape[0]} draws")

# K per solar mass of projected companion mass, at this period and eccentricity:
# the link from an amplitude to a mass function (quickstart 4).
k_per_msun = semi_amplitude_kms(mass_msun=1.0, period_yr=P_YR, ecc=truth.e, M_total_msun=truth.M_total_msun)
print(f"K per Msun of m2 sin i at this shape: {k_per_msun:.2f} km/s")

## Astrometry: nine columns, one solve

`ti_design_matrix` builds [A, B, F, G, Δα*, Δδ, μα*, μδ, ϖ] at a shape (the
parallax column is Gaia's own factor); `linear_solve_ti` solves it.
`ti_amplitude_chains` names the columns so `ti_to_kepler` can turn A, B, F, G
draws into the photocentre size and the angles.

In [ ]:
from orblet import (ti_design_matrix, linear_solve_ti, best_linear_params_ti,
                            ti_amplitude_chains, ti_to_kepler)

Xti = ti_design_matrix(t_mjd, psi, pf, f_per_day=1.0 / truth.P_days, ecc=truth.e, tau=TAU,
                       epoch_ref_mjd=EPOCH_REF)
sol_ti = linear_solve_ti(d_obs, sigma, Xti)
print("design matrix shape:", Xti.shape)
print("beta = [A, B, F, G, dra, ddec, pmra, pmdec, plx]:", np.round(sol_ti.beta, 3))
print(f"parallax {sol_ti.beta[8]:.3f} mas (truth {truth.parallax_mas:.3f}); chi2/dof = {sol_ti.chi2:.1f}/{sol_ti.dof}")

blp_ti = best_linear_params_ti(truth.P_days, truth.e, TAU, epochs_mjd=t_mjd, scan_angle=psi,
                               parallax_factor_al=pf, centroid_pos=d_obs, centroid_pos_err=sigma,
                               epoch_ref_mjd=EPOCH_REF, draw=500)
chains = ti_to_kepler(ti_amplitude_chains(blp_ti.draws))
print(f"a_phot = {np.median(chains['a_phot_mas']):.3f} +/- {np.std(chains['a_phot_mas']):.3f} mas (truth {truth.a_phot_mas:.3f})")
print(f"i = {np.degrees(np.median(chains['inc_rad'])):.1f} deg (truth {np.degrees(truth.i_rad):.1f}; the mirror i -> 180 - i is NOT resolved by astrometry alone)")

## The no-orbit model and the acceleration block

`fit_astrometric_5param` is the standard Gaia single-star solution, closed form.
`acceleration_columns` adds the two sky-plane acceleration columns (the ½ t²
convention, same reference epoch as the proper-motion columns — that shared
origin is load-bearing). Stacking blocks is how you build your own model.

In [ ]:
from orblet import astrometric_5param_design_matrix, fit_astrometric_5param, acceleration_columns

sol5 = fit_astrometric_5param(astro, epoch_ref_mjd=EPOCH_REF)
X5 = astrometric_5param_design_matrix(t_mjd, psi, pf, epoch_ref_mjd=EPOCH_REF)
chi2_5 = float(np.sum(((d_obs - X5 @ sol5.params) / sigma) ** 2))
print("5-parameter solution [dra, ddec, pmra, pmdec, plx]:", np.round(sol5.params, 3))
print(f"chi2/dof with NO orbit: {chi2_5:.1f}/{d_obs.size - 5} (the orbit is in the residuals)")

X7 = np.hstack([X5, acceleration_columns(t_mjd, psi, epoch_ref_mjd=EPOCH_REF)])
sol7 = linear_solve_ti(d_obs, sigma, X7)
print("7-parameter (5 + acceleration [a_ra*, a_dec] in mas/yr^2) solution:", np.round(sol7.beta, 3))
print(f"chi2/dof with acceleration: {sol7.chi2:.1f}/{sol7.dof}; with the orbit: {sol_ti.chi2:.1f}/{sol_ti.dof}")

## The three numbers a solve returns, and how to combine them

`logL_marginal` is the flat-prior marginal likelihood at the shape: the
likelihood integrated over the amplitudes with no prior, computed from the
data and the design matrix alone. It means the same thing whether or not you
pass an amplitude prior. A Gaussian `beta_prior` changes two other things:
`beta` and `cov` become the prior-regularised posterior, and `log_evidence`
becomes available, the likelihood integrated over the amplitudes *weighted by
that prior*, with every constant kept.

Rules: sum the **same** score across independent channels at one shape (both
`logL_marginal`, or both `log_evidence`); never add one kind to the other;
compare shapes by differences of logs, never by sums across shapes; and use
`log_evidence`, not the ranking score, to compare models with different
numbers of amplitudes.

In [ ]:
class GaussianBeta:
    """Any object exposing .mean and .cov is an amplitude prior for the solves."""
    def __init__(self, mean, cov):
        self.mean = np.asarray(mean, dtype=float)
        self.cov = np.asarray(cov, dtype=float)

prior = GaussianBeta(mean=[0.0, 0.0, 0.0], cov=np.diag([50.0, 50.0, 50.0]) ** 2)
sol_flat = linear_solve_rv(rv, rv_err, X)
sol_prior = linear_solve_rv(rv, rv_err, X, beta_prior=prior)
print(f"flat path:  logL_marginal = {sol_flat.logL_marginal:.3f}, log_evidence = {sol_flat.log_evidence}")
print(f"prior path: logL_marginal = {sol_prior.logL_marginal:.3f} (same number: the prior never enters it), "
      f"log_evidence = {sol_prior.log_evidence:.3f}")
print("amplitudes, flat vs regularised:", np.round(sol_flat.beta, 3), np.round(sol_prior.beta, 3))

Next: `03_period_search.ipynb` — searching the shape, and saying how significant
a peak is.